# Azure CLI Cheatsheet 

## Table of Contents
1. [Azure CLI Setup](#azure-cli-setup)
2. [Resource Groups](#resource-groups)
3. [Virtual Machines](#virtual-machines)
4. [Storage Accounts](#storage-accounts)
5. [AKS](#aks)
6. [App Service](#app-service)
7. [Azure SQL](#azure-sql)
8. [Virtual Networks](#virtual-networks)
9. [Container Registry](#container-registry)
10. [Key Vault](#key-vault)
11. [Azure Functions](#azure-functions)
12. [ARM Templates & Bicep](#arm-templates--bicep)
13. [Monitoring](#monitoring)
14. [Interview Scenarios](#interview-scenarios)

---

## Azure CLI Setup

### 1. Install Azure CLI


In [ ]:
# macOS
brew install azure-cli

# Linux (Debian/Ubuntu)
curl -sL https://aka.ms/InstallAzureCLIDeb | sudo bash

# Verify
az --version



### 2. Login & Subscription


In [ ]:
# Login
az login

# Login with service principal
az login --service-principal -u CLIENT_ID -p CLIENT_SECRET --tenant TENANT_ID

# List subscriptions
az account list --output table

# Set active subscription
az account set --subscription "My Subscription"

# Show current subscription
az account show



### 3. Configuration


In [ ]:
# Set default location
az configure --defaults location=eastus

# Set default resource group
az configure --defaults group=myResourceGroup

# List configuration
az configure --list-defaults

# Set output format
az config set core.output=table  # or json, yaml, jsonc, tsv



---

## Resource Groups

### 4. Manage Resource Groups


In [ ]:
# Create resource group
az group create --name myResourceGroup --location eastus

# List resource groups
az group list --output table

# Show resource group
az group show --name myResourceGroup

# Delete resource group
az group delete --name myResourceGroup --yes --no-wait

# List resources in group
az resource list --resource-group myResourceGroup --output table

# Export template
az group export --name myResourceGroup > template.json



---

## Virtual Machines

### 5. Create VM


In [ ]:
# Create Linux VM
az vm create \
  --resource-group myResourceGroup \
  --name myVM \
  --image UbuntuLTS \
  --admin-username azureuser \
  --generate-ssh-keys \
  --size Standard_B2s

# Create Windows VM
az vm create \
  --resource-group myResourceGroup \
  --name myWindowsVM \
  --image Win2019Datacenter \
  --admin-username azureuser \
  --admin-password MyPassword123!

# With custom data (cloud-init)
az vm create \
  --resource-group myResourceGroup \
  --name myVM \
  --image UbuntuLTS \
  --custom-data cloud-init.txt \
  --generate-ssh-keys



### 6. Manage VMs


In [ ]:
# List VMs
az vm list --output table
az vm list --resource-group myResourceGroup

# Show VM details
az vm show --resource-group myResourceGroup --name myVM

# Start VM
az vm start --resource-group myResourceGroup --name myVM

# Stop (deallocate)
az vm deallocate --resource-group myResourceGroup --name myVM

# Restart VM
az vm restart --resource-group myResourceGroup --name myVM

# Delete VM
az vm delete --resource-group myResourceGroup --name myVM --yes



### 7. VM Operations


In [ ]:
# Get public IP
az vm list-ip-addresses --resource-group myResourceGroup --name myVM --output table

# Resize VM
az vm resize --resource-group myResourceGroup --name myVM --size Standard_D4s_v3

# Run command on VM
az vm run-command invoke \
  --resource-group myResourceGroup \
  --name myVM \
  --command-id RunShellScript \
  --scripts "sudo apt-get update && sudo apt-get install -y nginx"

# Open port
az vm open-port --resource-group myResourceGroup --name myVM --port 80



### 8. VM Images


In [ ]:
# List marketplace images
az vm image list --output table
az vm image list --publisher Canonical --output table

# List all images (takes time)
az vm image list --all --publisher Microsoft --output table

# Create custom image
az vm deallocate --resource-group myResourceGroup --name myVM
az vm generalize --resource-group myResourceGroup --name myVM
az image create \
  --resource-group myResourceGroup \
  --name myImage \
  --source myVM



---

## Storage Accounts

### 9. Storage Account Operations


In [ ]:
# Create storage account
az storage account create \
  --name mystorageaccount \
  --resource-group myResourceGroup \
  --location eastus \
  --sku Standard_LRS \
  --kind StorageV2

# List storage accounts
az storage account list --output table

# Get connection string
az storage account show-connection-string \
  --name mystorageaccount \
  --resource-group myResourceGroup

# Get account keys
az storage account keys list \
  --resource-group myResourceGroup \
  --account-name mystorageaccount

# Delete storage account
az storage account delete \
  --name mystorageaccount \
  --resource-group myResourceGroup --yes



### 10. Blob Storage


In [ ]:
# Set account key as variable
ACCOUNT_KEY=$(az storage account keys list --resource-group myResourceGroup --account-name mystorageaccount --query '[0].value' -o tsv)

# Create container
az storage container create \
  --name mycontainer \
  --account-name mystorageaccount \
  --account-key $ACCOUNT_KEY

# Upload blob
az storage blob upload \
  --container-name mycontainer \
  --name myblob \
  --file /path/to/file.txt \
  --account-name mystorageaccount \
  --account-key $ACCOUNT_KEY

# List blobs
az storage blob list \
  --container-name mycontainer \
  --account-name mystorageaccount \
  --account-key $ACCOUNT_KEY --output table

# Download blob
az storage blob download \
  --container-name mycontainer \
  --name myblob \
  --file downloaded.txt \
  --account-name mystorageaccount \
  --account-key $ACCOUNT_KEY

# Delete blob
az storage blob delete \
  --container-name mycontainer \
  --name myblob \
  --account-name mystorageaccount \
  --account-key $ACCOUNT_KEY



### 11. File Shares


In [ ]:
# Create file share
az storage share create \
  --name myshare \
  --account-name mystorageaccount \
  --account-key $ACCOUNT_KEY

# Upload file to share
az storage file upload \
  --share-name myshare \
  --source /path/to/file.txt \
  --account-name mystorageaccount \
  --account-key $ACCOUNT_KEY

# Mount file share
# On Linux:
sudo mkdir /mnt/myshare
sudo mount -t cifs //mystorageaccount.file.core.windows.net/myshare /mnt/myshare -o vers=3.0,username=mystorageaccount,password=$ACCOUNT_KEY,dir_mode=0777,file_mode=0777



---

## AKS

### 12. AKS Cluster Operations


In [ ]:
# Create AKS cluster
az aks create \
  --resource-group myResourceGroup \
  --name myAKSCluster \
  --node-count 3 \
  --node-vm-size Standard_D2s_v3 \
  --generate-ssh-keys \
  --enable-addons monitoring

# List AKS clusters
az aks list --output table

# Get credentials
az aks get-credentials \
  --resource-group myResourceGroup \
  --name myAKSCluster

# Show cluster details
az aks show --resource-group myResourceGroup --name myAKSCluster

# Delete cluster
az aks delete --resource-group myResourceGroup --name myAKSCluster --yes --no-wait



### 13. AKS Node Operations


In [ ]:
# Scale cluster
az aks scale \
  --resource-group myResourceGroup \
  --name myAKSCluster \
  --node-count 5

# Upgrade cluster
az aks upgrade \
  --resource-group myResourceGroup \
  --name myAKSCluster \
  --kubernetes-version 1.27.0

# Get available upgrades
az aks get-upgrades \
  --resource-group myResourceGroup \
  --name myAKSCluster --output table

# Start stopped cluster
az aks start --resource-group myResourceGroup --name myAKSCluster

# Stop cluster
az aks stop --resource-group myResourceGroup --name myAKSCluster



### 14. Node Pools


In [ ]:
# Add node pool
az aks nodepool add \
  --resource-group myResourceGroup \
  --cluster-name myAKSCluster \
  --name mynodepool \
  --node-count 2 \
  --node-vm-size Standard_D4s_v3

# List node pools
az aks nodepool list \
  --resource-group myResourceGroup \
  --cluster-name myAKSCluster --output table

# Scale node pool
az aks nodepool scale \
  --resource-group myResourceGroup \
  --cluster-name myAKSCluster \
  --name mynodepool \
  --node-count 3

# Delete node pool
az aks nodepool delete \
  --resource-group myResourceGroup \
  --cluster-name myAKSCluster \
  --name mynodepool



---

## App Service

### 15. App Service Plan


In [ ]:
# Create app service plan
az appservice plan create \
  --name myAppServicePlan \
  --resource-group myResourceGroup \
  --sku B1 \
  --is-linux

# List plans
az appservice plan list --output table

# Delete plan
az appservice plan delete \
  --name myAppServicePlan \
  --resource-group myResourceGroup --yes



### 16. Web Apps


In [ ]:
# Create web app
az webapp create \
  --name myWebApp \
  --resource-group myResourceGroup \
  --plan myAppServicePlan \
  --runtime "NODE|14-lts"

# List web apps
az webapp list --output table

# Deploy from Git
az webapp deployment source config \
  --name myWebApp \
  --resource-group myResourceGroup \
  --repo-url https://github.com/user/repo \
  --branch main \
  --manual-integration

# Deploy zip file
az webapp deployment source config-zip \
  --name myWebApp \
  --resource-group myResourceGroup \
  --src app.zip

# Set environment variables
az webapp config appsettings set \
  --name myWebApp \
  --resource-group myResourceGroup \
  --settings KEY1=value1 KEY2=value2

# Restart web app
az webapp restart --name myWebApp --resource-group myResourceGroup

# Delete web app
az webapp delete --name myWebApp --resource-group myResourceGroup



### 17. Deployment Slots


In [ ]:
# Create deployment slot
az webapp deployment slot create \
  --name myWebApp \
  --resource-group myResourceGroup \
  --slot staging

# Swap slots
az webapp deployment slot swap \
  --name myWebApp \
  --resource-group myResourceGroup \
  --slot staging

# Delete slot
az webapp deployment slot delete \
  --name myWebApp \
  --resource-group myResourceGroup \
  --slot staging



---

## Azure SQL

### 18. SQL Server


In [ ]:
# Create SQL server
az sql server create \
  --name myserver \
  --resource-group myResourceGroup \
  --location eastus \
  --admin-user myadmin \
  --admin-password MyPassword123!

# List SQL servers
az sql server list --output table

# Configure firewall
az sql server firewall-rule create \
  --resource-group myResourceGroup \
  --server myserver \
  --name AllowMyIP \
  --start-ip-address 1.2.3.4 \
  --end-ip-address 1.2.3.4

# Delete SQL server
az sql server delete --name myserver --resource-group myResourceGroup --yes



### 19. SQL Database


In [ ]:
# Create database
az sql db create \
  --resource-group myResourceGroup \
  --server myserver \
  --name mydatabase \
  --service-objective S0

# List databases
az sql db list --resource-group myResourceGroup --server myserver --output table

# Scale database
az sql db update \
  --resource-group myResourceGroup \
  --server myserver \
  --name mydatabase \
  --service-objective S1

# Delete database
az sql db delete \
  --resource-group myResourceGroup \
  --server myserver \
  --name mydatabase --yes



---

## Virtual Networks

### 20. VNet Operations


In [ ]:
# Create virtual network
az network vnet create \
  --resource-group myResourceGroup \
  --name myVNet \
  --address-prefix 10.0.0.0/16 \
  --subnet-name mySubnet \
  --subnet-prefix 10.0.1.0/24

# List VNets
az network vnet list --output table

# Create subnet
az network vnet subnet create \
  --resource-group myResourceGroup \
  --vnet-name myVNet \
  --name mySubnet2 \
  --address-prefix 10.0.2.0/24

# Delete VNet
az network vnet delete --resource-group myResourceGroup --name myVNet



### 21. Network Security Groups


In [ ]:
# Create NSG
az network nsg create \
  --resource-group myResourceGroup \
  --name myNSG

# Create NSG rule
az network nsg rule create \
  --resource-group myResourceGroup \
  --nsg-name myNSG \
  --name allow-http \
  --protocol tcp \
  --priority 100 \
  --destination-port-range 80 \
  --access Allow

# List NSG rules
az network nsg rule list \
  --resource-group myResourceGroup \
  --nsg-name myNSG --output table

# Associate NSG with subnet
az network vnet subnet update \
  --resource-group myResourceGroup \
  --vnet-name myVNet \
  --name mySubnet \
  --network-security-group myNSG



### 22. Load Balancer


In [ ]:
# Create public IP
az network public-ip create \
  --resource-group myResourceGroup \
  --name myPublicIP \
  --sku Standard

# Create load balancer
az network lb create \
  --resource-group myResourceGroup \
  --name myLoadBalancer \
  --sku Standard \
  --public-ip-address myPublicIP \
  --frontend-ip-name myFrontEnd \
  --backend-pool-name myBackEndPool

# Create health probe
az network lb probe create \
  --resource-group myResourceGroup \
  --lb-name myLoadBalancer \
  --name myHealthProbe \
  --protocol tcp \
  --port 80

# Create load balancer rule
az network lb rule create \
  --resource-group myResourceGroup \
  --lb-name myLoadBalancer \
  --name myHTTPRule \
  --protocol tcp \
  --frontend-port 80 \
  --backend-port 80 \
  --frontend-ip-name myFrontEnd \
  --backend-pool-name myBackEndPool \
  --probe-name myHealthProbe



---

## Container Registry

### 23. ACR Operations


In [ ]:
# Create container registry
az acr create \
  --resource-group myResourceGroup \
  --name myregistry \
  --sku Basic

# List registries
az acr list --output table

# Login to registry
az acr login --name myregistry

# Build image in ACR
az acr build \
  --registry myregistry \
  --image myapp:v1 .

# List images
az acr repository list --name myregistry --output table

# List tags
az acr repository show-tags --name myregistry --repository myapp --output table

# Delete image
az acr repository delete \
  --name myregistry \
  --image myapp:v1 --yes

# Import image
az acr import \
  --name myregistry \
  --source docker.io/library/nginx:latest \
  --image nginx:latest



---

## Key Vault

### 24. Key Vault Operations


In [ ]:
# Create key vault
az keyvault create \
  --name mykeyvault \
  --resource-group myResourceGroup \
  --location eastus

# Set secret
az keyvault secret set \
  --vault-name mykeyvault \
  --name db-password \
  --value SuperSecret123!

# Get secret
az keyvault secret show \
  --vault-name mykeyvault \
  --name db-password

# List secrets
az keyvault secret list --vault-name mykeyvault --output table

# Delete secret
az keyvault secret delete \
  --vault-name mykeyvault \
  --name db-password

# Set access policy
az keyvault set-policy \
  --name mykeyvault \
  --object-id OBJECT_ID \
  --secret-permissions get list



---

## Azure Functions

### 25. Function App


In [ ]:
# Create function app
az functionapp create \
  --resource-group myResourceGroup \
  --consumption-plan-location eastus \
  --runtime python \
  --runtime-version 3.9 \
  --functions-version 4 \
  --name myfunctionapp \
  --storage-account mystorageaccount

# List function apps
az functionapp list --output table

# Deploy function (from local)
func azure functionapp publish myfunctionapp

# Set app settings
az functionapp config appsettings set \
  --name myfunctionapp \
  --resource-group myResourceGroup \
  --settings KEY=value

# Show logs
az functionapp log tail \
  --name myfunctionapp \
  --resource-group myResourceGroup

# Delete function app
az functionapp delete \
  --name myfunctionapp \
  --resource-group myResourceGroup



---

## ARM Templates & Bicep

### 26. ARM Templates


In [ ]:
# Validate template
az deployment group validate \
  --resource-group myResourceGroup \
  --template-file template.json \
  --parameters parameters.json

# Deploy template
az deployment group create \
  --resource-group myResourceGroup \
  --template-file template.json \
  --parameters parameters.json

# What-if deployment
az deployment group what-if \
  --resource-group myResourceGroup \
  --template-file template.json

# List deployments
az deployment group list --resource-group myResourceGroup --output table

# Show deployment
az deployment group show \
  --resource-group myResourceGroup \
  --name deploymentName



### 27. Bicep


In [ ]:
# Build Bicep to ARM
az bicep build --file main.bicep

# Decompile ARM to Bicep
az bicep decompile --file template.json

# Deploy Bicep
az deployment group create \
  --resource-group myResourceGroup \
  --template-file main.bicep



---

## Monitoring

### 28. Azure Monitor


In [ ]:
# Enable diagnostic logs
az monitor diagnostic-settings create \
  --name myDiagnosticSetting \
  --resource /subscriptions/.../resourceGroups/myRG/providers/Microsoft.Compute/virtualMachines/myVM \
  --logs '[{"category": "Administrative","enabled": true}]' \
  --metrics '[{"category": "AllMetrics","enabled": true}]' \
  --storage-account mystorageaccount

# Query logs
az monitor log-analytics query \
  --workspace WORKSPACE_ID \
  --analytics-query "AzureActivity | where TimeGenerated > ago(1h)"



### 29. Alerts


In [ ]:
# Create action group
az monitor action-group create \
  --name myActionGroup \
  --resource-group myResourceGroup \
  --short-name myAG

# Create metric alert
az monitor metrics alert create \
  --name high-cpu \
  --resource-group myResourceGroup \
  --scopes /subscriptions/.../resourceGroups/myRG/providers/Microsoft.Compute/virtualMachines/myVM \
  --condition "avg Percentage CPU > 80" \
  --description "Alert when CPU exceeds 80%" \
  --action myActionGroup

# List alerts
az monitor metrics alert list --resource-group myResourceGroup --output table



---

## Interview Scenarios

### Scenario 1: Deploy 3-Tier Web Application
**Question**: Deploy web app with Azure SQL backend and Azure Storage.



In [ ]:
# Create resource group
az group create --name app-rg --location eastus

# Create storage account
az storage account create \
  --name appstorage123 \
  --resource-group app-rg \
  --sku Standard_LRS

# Create SQL server and database
az sql server create \
  --name appserver123 \
  --resource-group app-rg \
  --admin-user sqladmin \
  --admin-password Password123!

az sql db create \
  --resource-group app-rg \
  --server appserver123 \
  --name appdb \
  --service-objective S0

# Configure firewall
az sql server firewall-rule create \
  --resource-group app-rg \
  --server appserver123 \
  --name AllowAzureServices \
  --start-ip-address 0.0.0.0 \
  --end-ip-address 0.0.0.0

# Create app service plan
az appservice plan create \
  --name app-plan \
  --resource-group app-rg \
  --sku B1 \
  --is-linux

# Create web app
az webapp create \
  --name mywebapp123 \
  --resource-group app-rg \
  --plan app-plan \
  --runtime "NODE|14-lts"

# Set connection strings
STORAGE_CONN=$(az storage account show-connection-string --name appstorage123 --resource-group app-rg --query connectionString -o tsv)
SQL_CONN="Server=tcp:appserver123.database.windows.net,1433;Database=appdb;User ID=sqladmin;Password=Password123!;"

az webapp config connection-string set \
  --name mywebapp123 \
  --resource-group app-rg \
  --connection-string-type SQLAzure \
  --settings DefaultConnection="$SQL_CONN"

az webapp config appsettings set \
  --name mywebapp123 \
  --resource-group app-rg \
  --settings STORAGE_CONNECTION_STRING="$STORAGE_CONN"



### Scenario 2: AKS with ACR Integration
**Question**: Deploy AKS cluster integrated with Azure Container Registry.



In [ ]:
# Create resource group
az group create --name k8s-rg --location eastus

# Create ACR
az acr create \
  --resource-group k8s-rg \
  --name myregistry123 \
  --sku Basic

# Create AKS with ACR integration
az aks create \
  --resource-group k8s-rg \
  --name myakscluster \
  --node-count 2 \
  --attach-acr myregistry123 \
  --generate-ssh-keys

# Get credentials
az aks get-credentials --resource-group k8s-rg --name myakscluster

# Build and push image to ACR
az acr build --registry myregistry123 --image myapp:v1 .

# Deploy to AKS
kubectl create deployment myapp --image=myregistry123.azurecr.io/myapp:v1
kubectl expose deployment myapp --type=LoadBalancer --port=80



### Scenario 3: Implement Blue-Green Deployment
**Question**: Set up blue-green deployment for web app.



In [ ]:
# Create web app
az webapp create \
  --name myapp123 \
  --resource-group myResourceGroup \
  --plan myAppServicePlan

# Create staging slot (green)
az webapp deployment slot create \
  --name myapp123 \
  --resource-group myResourceGroup \
  --slot staging

# Deploy to staging
az webapp deployment source config-zip \
  --name myapp123 \
  --resource-group myResourceGroup \
  --slot staging \
  --src app-v2.zip

# Test staging
curl https://myapp123-staging.azurewebsites.net

# Swap to production (zero downtime)
az webapp deployment slot swap \
  --name myapp123 \
  --resource-group myResourceGroup \
  --slot staging

# Rollback if needed
az webapp deployment slot swap \
  --name myapp123 \
  --resource-group myResourceGroup \
  --slot staging



### Scenario 4: Disaster Recovery Setup
**Question**: Implement multi-region DR for critical application.



In [ ]:
# Primary region (East US)
az group create --name app-east --location eastus

az sql server create \
  --name sqleast123 \
  --resource-group app-east \
  --admin-user sqladmin \
  --admin-password Password123!

az sql db create \
  --resource-group app-east \
  --server sqleast123 \
  --name appdb \
  --service-objective S1

# Secondary region (West US)
az group create --name app-west --location westus

az sql server create \
  --name sqlwest123 \
  --resource-group app-west \
  --admin-user sqladmin \
  --admin-password Password123!

# Set up geo-replication
az sql db replica create \
  --resource-group app-east \
  --server sqleast123 \
  --name appdb \
  --partner-server sqlwest123 \
  --partner-resource-group app-west

# Failover to secondary
az sql db replica set-primary \
  --resource-group app-west \
  --server sqlwest123 \
  --name appdb

# Set up Traffic Manager for apps
az network traffic-manager profile create \
  --name app-tm \
  --resource-group app-east \
  --routing-method Priority \
  --unique-dns-name myapp123

az network traffic-manager endpoint create \
  --resource-group app-east \
  --profile-name app-tm \
  --name east-endpoint \
  --type azureEndpoints \
  --target-resource-id /subscriptions/.../resourceGroups/app-east/providers/Microsoft.Web/sites/myapp-east \
  --priority 1

az network traffic-manager endpoint create \
  --resource-group app-east \
  --profile-name app-tm \
  --name west-endpoint \
  --type azureEndpoints \
  --target-resource-id /subscriptions/.../resourceGroups/app-west/providers/Microsoft.Web/sites/myapp-west \
  --priority 2



### Scenario 5: Cost Optimization
**Question**: Identify and optimize Azure costs.



In [ ]:
# Find unattached disks
az disk list --query "[?managedBy==null].[name,resourceGroup,diskSizeGb]" --output table

# Delete unattached disks
for disk in $(az disk list --query "[?managedBy==null].id" -o tsv); do
  az disk delete --ids $disk --yes --no-wait
done

# Find unused public IPs
az network public-ip list --query "[?ipConfiguration==null].[name,resourceGroup]" --output table

# Delete unused public IPs
for ip in $(az network public-ip list --query "[?ipConfiguration==null].id" -o tsv); do
  az network public-ip delete --ids $ip
done

# Resize over-provisioned VMs
# Check CPU metrics first
az monitor metrics list \
  --resource /subscriptions/.../resourceGroups/myRG/providers/Microsoft.Compute/virtualMachines/myVM \
  --metric "Percentage CPU" \
  --start-time 2024-01-01T00:00:00Z \
  --end-time 2024-01-07T23:59:59Z

# If avg CPU &lt; 20%, downsize
az vm deallocate --resource-group myRG --name myVM
az vm resize --resource-group myRG --name myVM --size Standard_B1s
az vm start --resource-group myRG --name myVM

# Use Azure Advisor
az advisor recommendation list --output table



---

## Quick Reference

### Common Patterns


In [ ]:
# JMESPath queries
--query "[*].[name,location]" --output table
--query "[?contains(name,'prod')]"

# Output formats
--output table   # Human readable
--output json    # Default
--output yaml    # YAML format
--output tsv     # Tab separated

# Async operations
--no-wait        # Don't wait for completion

# Skip confirmation
--yes            # Auto-confirm deletions



### Best Practices


In [ ]:
# Use tags for organization
az group create --name myRG --location eastus --tags Environment=Production CostCenter=IT

# Use managed identities instead of passwords
az vm identity assign --resource-group myRG --name myVM

# Enable diagnostic logs for all resources
# Use Azure Policy to enforce standards
# Implement resource locks on critical resources



---

**Prepared for**: Lead DevOps Architect Interview  
**Last Updated**: February 15, 2026  
**Total Commands**: 100+ Azure CLI commands
